# 2. Put the material into native FEM

**Goal:** turn lesson 1's density into mesh energy, its displacement gradient, and stiffness.
Only this lesson introduces the material adapter and assembly. The geometry helper
creates two tetrahedra with five vertices; it contains no material or solver logic.

You now have two different derivatives: stress $\partial W/\partial F$ at a material
point, and $g=\partial E/\partial u$ for the whole mesh. Native FEM connects them.
Code is repeated where needed so this notebook runs from a fresh kernel.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
%matplotlib inline
from litepgo import dof, energy, fem, solver
from litepgo.torch import TorchVolumetricMaterial, response
from common import make_geometry

torch.set_num_threads(1)
dtype = torch.float64

## One small adapter around the same law

Subclass `TorchVolumetricMaterial` and provide `num_parameters` and `W`.
The signature adds `Fp`; here every $F_p=I$. The multiplicative split and
$\det F_p$ factor preserve the reference-volume convention if plasticity is used later.

`W` receives `(N,3,3)`, `(N,3,3)`, `(N,2)` and returns `(N,)`. The base class computes
requested derivatives automatically. Invalid physical parameters return NaN so native
FEM can reject the evaluation.

In [ ]:
class TorchNeoHookean(TorchVolumetricMaterial):
    num_parameters = 2

    @staticmethod
    def W(F, Fp, theta):
        Fe = F @ torch.linalg.inv(Fp)
        lam, mu = theta.unbind(-1)
        log_J = torch.linalg.det(Fe).log()
        psi = 0.5 * mu * (Fe.square().sum((-2, -1)) - 3) - mu * log_J + 0.5 * lam * log_J.square()
        W = torch.linalg.det(Fp) * psi
        return torch.where((lam >= 0) & (mu > 0), W, torch.full_like(W, float('nan')))

## Select material, displacement basis, and quadrature

`TetP1Basis` means linear tetrahedral displacement interpolation; centroid quadrature
uses one material point per element. The same geometry will be used in lessons 3 and 4.
A constant `AffineResponse` samples the same parameter vector everywhere.

In [ ]:
X, topology, mapping, geometry, plastic = make_geometry()
key = (TorchNeoHookean, fem.TetP1Basis, fem.TetCentroidQuadrature)
truth = np.array([2., 3.])
fields = [energy.AffineResponse(np.zeros((2, 3)), truth)] * topology.num_cells
potential = fem.VolumetricFemEnergy[key](mapping, geometry, plastic, fields)
strain = np.array([[.08, .03, 0], [0, -.02, .02], [.01, 0, .04]])
u = (X @ strain.T).ravel()
E, g, K = potential.eval(u, gradient=True, hessian=energy.HessianKind.Exact)
print('vertices / elements / displacement DOFs:', len(X), topology.num_cells, mapping.num_dofs)
print('E:', E, 'gradient shape:', g.shape, 'stiffness shape:', K.shape)

## Compare with native Neo-Hookean

Change only the material in the template key. Both implementations must give the
same $E$, $g$, and exact $K$. `g` is the **positive energy gradient**, not its negative.
The matrix is singular without boundary constraints; we are not solving equilibrium yet.

In [ ]:
native = fem.VolumetricFemEnergy[fem.NeoHookean, *key[1:]](mapping, geometry, plastic, fields)
En, gn, Kn = native.eval(u, gradient=True, hessian=energy.HessianKind.Exact)
np.testing.assert_allclose(E, En, rtol=1e-11, atol=1e-13)
np.testing.assert_allclose(g, gn, rtol=1e-10, atol=1e-12)
np.testing.assert_allclose(K.toarray(), Kn.toarray(), rtol=1e-9, atol=1e-11)
print('native comparison: energy, gradient, and Hessian passed')
direction = np.linspace(-.2, .3, len(u))
h = 1e-6
fd = (potential.eval_gradient(u + h * direction) - potential.eval_gradient(u - h * direction)) / (2 * h)
np.testing.assert_allclose(K @ direction, fd, rtol=1e-6, atol=1e-9)
print('stiffness directional check passed')

## See the prescribed deformation

This is an imposed displacement, not a solved one. Thin gray edges show rest geometry;
blue edges show the prescribed configuration.

In [ ]:
from itertools import combinations
cells = np.array([[0, 1, 2, 3], [1, 2, 3, 4]])
edges = sorted({tuple(sorted(edge)) for cell in cells for edge in combinations(cell, 2)})
fig = plt.figure(figsize=(5, 4), constrained_layout=True)
ax = fig.add_subplot(projection='3d')
for positions, color, label in [(X, 'gray', 'Rest'), (X + u.reshape(-1, 3), 'tab:blue', 'Prescribed')]:
    for i, edge in enumerate(edges):
        points = positions[list(edge)]
        ax.plot(*points.T, color=color, label=label if i == 0 else None)
ax.set(xlabel='x', ylabel='y', zlabel='z', title='Same two-tetrahedron mesh throughout lessons 2–4')
ax.legend()
plt.show()

## Try it

Change the shear entries in `strain`. Repeat both derivative checks. Change the
material parameter values and see which outputs change.

**Checkpoint:** Torch defines the density; native FEM handles geometry, quadrature,
chain rules, and assembly. Next: fit material parameters with this pose held fixed.